In [87]:
import cv2
import numpy as np
import subprocess
import time
import pygame

In [106]:
import json, numpy as np, cv2

with open("calibration.json") as f:
    cal = json.load(f)

H = np.array(cal["projector_homography"], dtype=np.float64)

In [107]:
print("det(H) =", np.linalg.det(H))   # should be roughly 1.0-3.0; very negative = mirrored

H_inv = np.linalg.inv(H)
for x, y in [(0, 0), (1920, 0), (1920, 1080), (0, 1080)]:
    p = cv2.perspectiveTransform(np.array([[[x, y]]], dtype=np.float32), H_inv)
    print(f"proj ({x},{y}) → cam ({p[0][0][0]:.0f}, {p[0][0][1]:.0f})")
    # These 4 camera coords should match the 4 spots you clicked in the original
    # calibration run. If they don't, the file is for a different camera position.

det(H) = 2.1840251652954406
proj (0,0) → cam (270, 93)
proj (1920,0) → cam (1553, 63)
proj (1920,1080) → cam (1627, 795)
proj (0,1080) → cam (273, 852)


In [121]:
cap = cv2.VideoCapture(0)              # force V4L2 backend
# cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1920)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
CAMERA_W = 3840
CAMERA_H = 2160
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_W)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_H)
cap.read()                                            # let OpenCV finish init
pygame.init()
screen = pygame.display.set_mode((0, 0), pygame.FULLSCREEN, display=1)
clock = pygame.time.Clock()
# # 1) manual mode MUST come first, separately, so the next write is honored
# subprocess.run(["v4l2-ctl", "-d", "/dev/video0",
#                 "-c", "auto_exposure=1"], check=True)

# # 2) now set the actual exposure time
# subprocess.run(["v4l2-ctl", "-d", "/dev/video0",
#                 "-c", "exposure_time_absolute=10"], check=True)
DICT = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_100)
PARAMS = cv2.aruco.DetectorParameters()
PARAMS.minMarkerPerimeterRate = 0.005    # default 0.03 — 6x lower
PARAMS.minMarkerDistanceRate  = 0.005
PARAMS.perspectiveRemovePixelPerCell = 4
DETECTOR = cv2.aruco.ArucoDetector(DICT, PARAMS)
time.sleep(0.2)
while True:
    # for event in pygame.event.get():
        # pass
    screen.fill((0, 0, 0)) 
    ret, frame = cap.read()
    if not ret: break

    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    corners, ids, rejected = DETECTOR.detectMarkers(gray)
    screen.fill((0, 0, 0))  
    if ids is not None:                        # black bg
        corners = [c / 2.0 for c in corners]
        for corner, mid in zip(corners, ids.flatten()):
            x, y = corner[0][0].astype(int)
            cv2.circle(frame, (x, y), 5, (255, 0, 0), -1)
            pt = np.array([[x, y]], dtype=np.float32).reshape(-1, 1, 2)
            # print(pt)
            proj_pt = cv2.perspectiveTransform(pt, H)
            # print(pt, proj_pt)
            px, py = int(proj_pt[0][0][0]), int(proj_pt[0][0][1])
            pygame.draw.circle(screen, (255, 0, 0),
                   (px,py),
                   5)
            # break
    # LOWER = np.array([  35, , 220], np.uint8)   # H, S, V
    # UPPER = np.array([ 85, 255, 255], np.uint8) 
    # hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    # gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    # mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,
    #                     cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))


    # _, mask = cv2.threshold(gray, , 255, cv2.THRESH_BINARY)
    # _, bright = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY)
    # mask = cv2.inRange(hsv, LOWER, UPPER)
    # mask = cv2.bitwise_and(mask, bright)
    # M = cv2.moments(mask, binaryImage=True)
    # if M["m00"] > 0:
    #     cx = M["m10"] // M["m00"]
    #     cy = M["m01"] // M["m00"]
    #     # print(cx, cy)
    #     cv2.circle(frame, (int(cx), int(cy)), 5, (255, 0, 0), -1)
    cv2.imshow("cam", frame)
    pygame.display.flip()
    clock.tick(60)
    if cv2.waitKey(1) & 0xFF == ord("q"): break
cap.release()
cv2.destroyAllWindows()
pygame.display.quit()
pygame.quit()

In [21]:
screen.get_width()

1024

In [22]:
screen.get_height()

768

In [ ]:
margin = 80
proj_pts = [
    [margin,        margin       ],   # top-left
    [proj_w-margin, margin       ],   # top-right
    [proj_w-margin, proj_h-margin],   # bottom-right
    [margin,        proj_h-margin],   # bottom-left
]


In [24]:
                         # black bg
pygame.draw.circle(screen, (255, 255, 255),
                   (screen.get_width()//2, screen.get_height()//2),
                   5)  # green, center, radius
# pygame.draw.circle(screen, (255, 0, 0), (100, 100), 30, 2)



pygame.display.flip()

In [82]:
pygame.display.quit()
pygame.quit()